# Експерименти


In [ ]:
!pip install -q pytorch-lightning wandb tokenizers

In [ ]:
import os, re, json, pickle, string, random
from typing import List
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

SEED     = 21
MAX_LEN  = 256
PAD, UNK = 0, 1
IGNORE   = -100
N_TRAIN  = 150_000
EPOCHS   = 3
BATCH    = 128

USE_WANDB     = True
WANDB_PROJECT = 'dl-3'
WANDB_ENTITY  = 'yrachynska-kyiv-school-of-economics'
WANDB_SECRET  = 'WANDB_API_KEY'

pl.seed_everything(SEED)
files = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs]
def find(name):
    found = [f for f in files if os.path.basename(f) == name]
    if not found:
        raise FileNotFoundError(f'{name} не знайдено в /kaggle/input. Додайте вихід ноутбука 03 як Input '
                                f'(і зробіть там Save Version). Зараз доступні файли: {[os.path.basename(f) for f in files][:20]}')
    return found[0]

if USE_WANDB:
    try:
        import wandb
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret(WANDB_SECRET))
        list(wandb.Api().projects(WANDB_ENTITY))
        print(f'W&B: підключено, проєкт {WANDB_ENTITY}/{WANDB_PROJECT}')
    except Exception as e:
        print('W&B НЕ підключено:', e, '\nПеревірте назву секрету, WANDB_ENTITY (адреса проєкту wandb.ai/<entity>/dl-3) та Internet on. Далі логуємо лише в CSV.')
        USE_WANDB = False

print('GPU:', torch.cuda.is_available())

In [ ]:
with open(find('prepared.pkl'), 'rb') as f:
    prepared = pickle.load(f)

val_text, lab_text, test_text = prepared['val_text'], prepared['lab_text'], prepared['test_text']
print({k: len(prepared['word'][k]['ids']) for k in ['big', 'val', 'lab', 'test']})
print('словник word:', len(prepared['word']['vocab']), '| словник bpe:', prepared['bpe']['vocab_size'])

In [ ]:
def hard_processing(text):
    text = re.sub(r"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d', '', text)
    text = re.sub(r'\b\w\b\s?', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def comp_metric(y_true: List[List[str]], y_pred: List[List[str]]):
    assert len(y_true) == len(y_pred)
    tp, fp, fn, p = 0.0, 0.0, 0.0, 0.0
    for t, pr in zip(y_true, y_pred):
        t = set([hard_processing(s) for s in t])
        pr = set([hard_processing(s) for s in pr])
        tp += len(t & pr); fp += len(pr - t); fn += len(t - pr); p += len(t)
    if tp + fp == 0:
        precision = 1.0 if p == 0 else 0.0
    else:
        precision = tp / (tp + fp)
    if tp + fn == 0:
        recall = 1.0 if p == 0 else 0.0
    else:
        recall = tp / (tp + fn)
    f1 = 0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return round(f1, 8)

EMOJI = re.compile('[\U0001F000-\U0001FFFF\u2600-\u27BF\uFE0F]')
POST_RULES = set()

def clean_span(s, rules):
    if 'newline' in rules:
        s = s.split('\n')[0]
    if 'emoji' in rules:
        s = EMOJI.sub('', s)
    s = s.strip()
    if 'strip' in rules:
        s = s.strip(' \t,.;:!?-–—"«»“”()')
    if 'letters' in rules and not re.search(r'[^\W\d_]', s):
        return None
    if 'upper' in rules and not any(w[0].isupper() for w in re.findall(r'\w+', s)):
        return None
    return s or None

def decode(text, offsets, probs, thr):
    is_loc = (1 - probs[:, 0]) > thr
    result, start, end = [], None, None
    for t, (t_start, t_end) in enumerate(offsets[:len(probs)]):
        new_loc = is_loc[t] and (probs[t, 1] >= probs[t, 2] or t == 0 or not is_loc[t - 1])
        if new_loc:
            if start is not None:
                result.append(text[start:end])
            start, end = t_start, t_end
        elif is_loc[t] and start is not None:
            end = t_end
        else:
            if start is not None:
                result.append(text[start:end])
            start, end = None, None
    if start is not None:
        result.append(text[start:end])
    if POST_RULES:
        result = [c for c in (clean_span(x, POST_RULES) for x in result) if c]
    return result

def score(probs_list, texts, offsets_list, golds, thr):
    preds = [decode(t, o, p, thr) for t, o, p in zip(texts, offsets_list, probs_list)]
    return comp_metric(golds, preds)

In [ ]:
class NERDataset(Dataset):
    def __init__(self, ids, labels):
        self.ids, self.labels = ids, labels
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return torch.tensor(self.ids[i][:MAX_LEN]), torch.tensor(self.labels[i][:MAX_LEN])

def collate(batch):
    ids, labels = zip(*batch)
    return {
        'input_ids': pad_sequence(ids, batch_first=True, padding_value=PAD),
        'labels':    pad_sequence(labels, batch_first=True, padding_value=IGNORE),
        'lengths':   torch.tensor([len(x) for x in ids]),
    }

@torch.no_grad()
def predict_probs(model, ids_list, batch_size=128):
    model.eval()
    chunks = []
    for d, ids in enumerate(ids_list):
        for s in range(0, max(len(ids), 1), MAX_LEN):
            chunks.append((d, s, ids[s:s + MAX_LEN]))
    pieces = {}
    for b in range(0, len(chunks), batch_size):
        batch = chunks[b:b + batch_size]
        x = pad_sequence([torch.tensor(c[2] if c[2] else [PAD]) for c in batch], batch_first=True, padding_value=PAD)
        lengths = torch.tensor([max(len(c[2]), 1) for c in batch])
        probs = torch.softmax(model(x.to(model.device), lengths), dim=-1).cpu().numpy()
        for (d, s, ids), p in zip(batch, probs):
            pieces[(d, s)] = p[:len(ids)]
    out = []
    for d, ids in enumerate(ids_list):
        parts = [pieces[(d, s)] for s in range(0, max(len(ids), 1), MAX_LEN)]
        out.append(np.concatenate(parts) if len(ids) else np.zeros((0, 3), dtype='float32'))
    return out

In [ ]:
class NERModel(pl.LightningModule):
    def __init__(self, vocab_size, emb_dim=200, hidden=256, layers=1, bidirectional=True, rnn='lstm',
                 dropout=0.3, lr=2e-3, optimizer='adam', scheduler='onecycle', loc_weight=2.0, emb_matrix=None):
        super().__init__()
        self.save_hyperparameters(ignore=['emb_matrix'])
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        if emb_matrix is not None:
            self.emb.weight.data.copy_(torch.tensor(emb_matrix))
        rnn_cls = nn.LSTM if rnn == 'lstm' else nn.GRU
        self.rnn = rnn_cls(emb_dim, hidden, num_layers=layers, batch_first=True, bidirectional=bidirectional,
                           dropout=dropout if layers > 1 else 0)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(hidden * (2 if bidirectional else 1), 3)
        self.register_buffer('class_weight', torch.tensor([1.0, loc_weight, loc_weight]))
        self.val_info = None

    def forward(self, ids, lengths):
        x = self.drop(self.emb(ids))
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.rnn(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=ids.size(1))
        return self.out(self.drop(out))

    def _loss(self, batch):
        logits = self(batch['input_ids'], batch['lengths'])
        return nn.functional.cross_entropy(logits.reshape(-1, 3), batch['labels'].reshape(-1),
                                           weight=self.class_weight, ignore_index=IGNORE)

    def training_step(self, batch, batch_idx):
        loss = self._loss(batch)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        self.log('val_loss', self._loss(batch), prog_bar=True)

    def on_before_optimizer_step(self, optimizer):
        norms = [p.grad.norm() ** 2 for p in self.parameters() if p.grad is not None]
        self.log('grad_norm', torch.sqrt(sum(norms)))

    def on_validation_epoch_end(self):
        if self.val_info is None or self.trainer.sanity_checking:
            return
        texts, offsets, golds, ids = self.val_info
        probs = predict_probs(self, ids)
        self.log('val_f1', score(probs, texts, offsets, golds, thr=0.5), prog_bar=True)

    def configure_optimizers(self):
        lr = self.hparams.lr
        if self.hparams.optimizer == 'adam':
            opt = torch.optim.Adam(self.parameters(), lr=lr)
        elif self.hparams.optimizer == 'adamw':
            opt = torch.optim.AdamW(self.parameters(), lr=lr, weight_decay=0.01)
        else:
            opt = torch.optim.SGD(self.parameters(), lr=lr, momentum=0.9)
        steps = self.trainer.estimated_stepping_batches
        if self.hparams.scheduler == 'onecycle':
            sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps)
        elif self.hparams.scheduler == 'cosine':
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
        else:
            return opt
        return {'optimizer': opt, 'lr_scheduler': {'scheduler': sch, 'interval': 'step'}}

In [ ]:
from tokenizers import Tokenizer

STREETS_GEN = ['Шевченка', 'Лесі Українки', 'Миру', 'Незалежності', 'Франка', 'Грушевського', 'Героїв України',
               'Гагаріна', 'Бандери', 'Хрещатик', 'Перемоги', 'Мазепи', 'Івасюка', 'Стуса', 'Сагайдачного']
STREETS_ADJ = ['Центральна', 'Соборна', 'Київська', 'Садова', 'Шкільна', 'Польова', 'Солом\'янська', 'Здолбунівська',
               'Покровська', 'Лісова', 'Набережна', 'Миколаївська']
ABBR = ['вул.', 'просп.', 'пров.', 'бул.']
FULL = ['вулиця', 'проспект', 'провулок', 'бульвар']
CITIES = ['Київ', 'Львів', 'Харків', 'Одеса', 'Дніпро', 'Біла Церква', 'Бровари', 'Ірпінь', 'Буча', 'Вишневе',
          'Фастів', 'Бориспіль', 'Чернігів', 'Вінниця', 'Житомир', 'Полтава']
LOCATIVES = ['Києві', 'Львові', 'Харкові', 'Одесі', 'Дніпрі', 'Броварах', 'Бучі', 'Ірпені', 'Фастові', 'Борисполі',
             'Василькові', 'Вишгороді', 'Обухові', 'Яготині', 'Білій Церкві', 'Чернігові', 'Вінниці', 'Житомирі',
             'Полтаві', 'Миронівці', 'Баришівці', 'Березані', 'Оболоні', 'Борщагівці', 'Позняках', 'Осокорках',
             'Теремках', 'Виноградарі', 'Нивках', 'Деснянському районі', 'Дарницькому районі']
TEMPLATES = [('Адреса: {A}.', True), ('Наша адреса — {A}.', True), ('{A} — гарне місце.', True),
             ('Звертайтесь: {C}, {A}.', True), ('Це сталося на {A}.', False),
             ('Магазин розташований на {A}.', False), ('Ми працюємо в місті {C}, {A}.', False),
             ('Це сталося в {L}.', False), ('Нещодавно в {L} знайшли загублений гаманець.', False)]

def make_address(rng, nominative):
    if rng.random() < 0.4:
        prefix = 'вулиця' if (nominative and rng.random() < 0.5) else 'вул.'
        street = rng.choice(STREETS_ADJ)
    else:
        prefix = rng.choice(FULL if (nominative and rng.random() < 0.5) else ABBR)
        street = rng.choice(STREETS_GEN)
    addr = f'{prefix} {street}'
    if rng.random() < 0.6:
        addr += f', {rng.randint(1, 150)}'
    return addr

def tokenize_words(text):
    toks = list(re.finditer(r"\w+|[^\w\s]", text))
    return [m.group() for m in toks], [(m.start(), m.end()) for m in toks]

def make_labels(offsets, spans):
    labels = [0] * len(offsets)
    for s_start, s_end in spans:
        first = True
        for i, (t_start, t_end) in enumerate(offsets):
            if t_start < s_end and t_end > s_start:
                labels[i] = 1 if first else 2
                first = False
    return labels

def make_synthetic(n, kind, rng):
    word_vocab = prepared['word']['vocab']
    if kind == 'bpe':
        if 'tokenizer_json' in prepared['bpe']:
            bpe = Tokenizer.from_str(prepared['bpe']['tokenizer_json'])
        else:
            bpe = Tokenizer.from_file(find('bpe.json'))
    ids_list, labels_list = [], []
    for _ in range(n):
        tpl, nominative = rng.choice(TEMPLATES)
        addr = make_address(rng, nominative)
        values = {'{A}': addr, '{C}': rng.choice(CITIES), '{L}': rng.choice(LOCATIVES)}
        text, spans = '', []
        for part in re.split(r'(\{[ACL]\})', tpl):
            if part in values:
                spans.append((len(text), len(text) + len(values[part])))
                text += values[part]
            else:
                text += part
        if kind == 'word':
            tokens, offsets = tokenize_words(text)
            ids = [word_vocab.get(w.lower(), UNK) for w in tokens]
        else:
            enc = bpe.encode(text)
            offsets = enc.offsets
            ids = [bpe.token_to_id(t) if bpe.token_to_id(t) is not None else UNK for t in enc.tokens]
        ids_list.append(ids)
        labels_list.append(make_labels(offsets, spans))
    return ids_list, labels_list

demo_ids, demo_labels = make_synthetic(3, 'word', random.Random(SEED))
print(demo_ids[0]); print(demo_labels[0])

In [ ]:
if "results" not in globals():
    results = []

def run_experiment(name, kind='word', rnn='lstm', layers=1, bidirectional=True, hidden=256, emb_dim=200,
                   lr=2e-3, optimizer='adam', scheduler='onecycle', dropout=0.3, loc_weight=2.0,
                   synth=0, epochs=EPOCHS, use_pretrained=False, n_train=None, use_lab=0):
    pl.seed_everything(SEED)
    data = prepared[kind]
    vocab_size = len(prepared['word']['vocab']) if kind == 'word' else prepared['bpe']['vocab_size']
    emb_matrix = prepared['word']['emb_matrix'] if (use_pretrained and kind == 'word') else None
    if emb_matrix is not None:
        emb_dim = emb_matrix.shape[1]

    idx = random.Random(SEED).sample(range(len(data['big']['ids'])), min(n_train or N_TRAIN, len(data['big']['ids'])))
    tr_ids    = [data['big']['ids'][i] for i in idx]
    tr_labels = [data['big']['labels'][i] for i in idx]
    if synth:
        s_ids, s_labels = make_synthetic(synth, kind, random.Random(SEED))
        tr_ids += s_ids; tr_labels += s_labels
    if use_lab:
        tr_ids += data['lab']['ids'] * use_lab
        tr_labels += data['lab']['labels'] * use_lab

    keep = [i for i, x in enumerate(tr_ids) if len(x) > 0]
    tr_ids, tr_labels = [tr_ids[i] for i in keep], [tr_labels[i] for i in keep]

    train_dl = DataLoader(NERDataset(tr_ids, tr_labels), batch_size=BATCH, shuffle=True, collate_fn=collate, num_workers=2)
    val_dl   = DataLoader(NERDataset(data['val']['ids'], data['val']['labels']), batch_size=256, collate_fn=collate)

    model = NERModel(vocab_size, emb_dim, hidden, layers, bidirectional, rnn, dropout, lr, optimizer, scheduler,
                     loc_weight, emb_matrix)
    model.val_info = (val_text, data['val']['offsets'], data['val']['locs'], data['val']['ids'])

    loggers = [pl.loggers.CSVLogger('/kaggle/working/logs', name=name)]
    if USE_WANDB:
        wandb_logger = pl.loggers.WandbLogger(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=name)
        loggers.append(wandb_logger)
    ckpt = pl.callbacks.ModelCheckpoint(dirpath=f'/kaggle/working/ckpt/{name}', monitor='val_f1', mode='max', save_top_k=1)
    trainer = pl.Trainer(max_epochs=epochs, accelerator='auto', devices=1, gradient_clip_val=1.0,
                         logger=loggers, callbacks=[ckpt], log_every_n_steps=50, enable_model_summary=False)
    trainer.fit(model, train_dl, val_dl)

    model = NERModel.load_from_checkpoint(ckpt.best_model_path).to('cuda' if torch.cuda.is_available() else 'cpu')
    probs = {'val':  predict_probs(model, data['val']['ids']),
             'lab':  predict_probs(model, data['lab']['ids']),
             'test': predict_probs(model, data['test']['ids'])}

    thrs = np.arange(0.1, 0.91, 0.05)
    val_scores = [score(probs['val'], val_text, data['val']['offsets'], data['val']['locs'], t) for t in thrs]
    thr = float(thrs[int(np.argmax(val_scores))])
    row = {'name': name, 'kind': kind, 'rnn': rnn, 'layers': layers, 'bidir': bidirectional, 'optimizer': optimizer,
           'scheduler': scheduler, 'synth': synth, 'thr': round(thr, 2),
           'val_f1@0.5': score(probs['val'], val_text, data['val']['offsets'], data['val']['locs'], 0.5),
           'val_f1': max(val_scores),
           'lab_f1': None if use_lab else score(probs['lab'], lab_text, data['lab']['offsets'], data['lab']['locs'], thr),
           'lab_in_train': use_lab,
           'LB': None}
    results.append(row)
    with open(f'/kaggle/working/preds_{name}.pkl', 'wb') as f:
        pickle.dump({'probs': probs, 'thr': thr, 'kind': kind}, f)
    if USE_WANDB:
        wandb_logger.experiment.summary.update({k: v for k, v in row.items() if k != 'LB' and v is not None})
        wandb.finish()
    print(row)
    return probs, thr

## Перевірка: чи модель вчиться

In [ ]:
small = prepared['word']['big']
dl = DataLoader(NERDataset(small['ids'][:64], small['labels'][:64]), batch_size=32, collate_fn=collate)
tiny = NERModel(len(prepared['word']['vocab']), lr=3e-3, scheduler='none', dropout=0.0)
trainer = pl.Trainer(max_epochs=100, accelerator='auto', devices=1, logger=False, enable_checkpointing=False,
                     enable_model_summary=False, enable_progress_bar=False)
trainer.fit(tiny, dl)
final = float(trainer.callback_metrics['train_loss'])
print('train_loss в кінці:', round(final, 4), '— OK' if final < 0.1 else '— ПІДОЗРІЛО: loss має впасти майже до нуля, шукайте помилку')

## Експерименти

In [ ]:
run_experiment('word_lstm_bi', kind='word', rnn='lstm')

In [ ]:
run_experiment('word_gru_bi', kind='word', rnn='gru')

In [ ]:
run_experiment('bpe_lstm_bi', kind='bpe', rnn='lstm')

In [ ]:
run_experiment('bpe_lstm_bi_2layers', kind='bpe', rnn='lstm', layers=2)

In [ ]:
run_experiment('bpe_lstm_uni', kind='bpe', rnn='lstm', bidirectional=False)

In [ ]:
run_experiment('bpe_lstm_adamw_cosine', kind='bpe', rnn='lstm', optimizer='adamw', scheduler='cosine')

In [ ]:
run_experiment('word_lstm_synth', kind='word', rnn='lstm', synth=20_000)

## Діагностика на lab

In [ ]:
def prf(golds, preds):
    tp = fp = fn = 0
    for g, p in zip(golds, preds):
        g = {hard_processing(s) for s in g}; p = {hard_processing(s) for s in p}
        tp += len(g & p); fp += len(p - g); fn += len(g - p)
    return round(tp / (tp + fp), 3) if tp + fp else 0, round(tp / (tp + fn), 3) if tp + fn else 0

def diagnose(name, show_errors=True):
    saved = pickle.load(open(f'/kaggle/working/preds_{name}.pkl', 'rb'))
    data, thr = prepared[saved['kind']], saved['thr']
    for split, texts in [('val', val_text), ('lab', lab_text)]:
        preds = [decode(t, o, p, thr) for t, o, p in zip(texts, data[split]['offsets'], saved['probs'][split])]
        print(f'{split}: (precision, recall) при порозі {thr:.2f} =', prf(data[split]['locs'], preds))
    print('lab F1 за порогами:',
          {round(float(t), 1): score(saved['probs']['lab'], lab_text, data['lab']['offsets'], data['lab']['locs'], t)
           for t in np.arange(0.1, 0.91, 0.1)})
    if show_errors:
        for t, o, p, gold in zip(lab_text, data['lab']['offsets'], saved['probs']['lab'], data['lab']['locs']):
            pred = decode(t, o, p, thr)
            gs, ps = {hard_processing(s) for s in gold}, {hard_processing(s) for s in pred}
            missed = [g for g in gold if hard_processing(g) not in ps]
            extra = [x for x in pred if hard_processing(x) not in gs]
            if missed or extra:
                print('пропущено:', missed, '| зайве:', extra)

diagnose('word_lstm_synth')

## Постобробка

In [ ]:
import itertools
RULES = ['newline', 'emoji', 'strip', 'letters', 'upper']

def eval_rules(name, rule_sets):
    global POST_RULES
    saved = pickle.load(open(f'/kaggle/working/preds_{name}.pkl', 'rb'))
    data, thr = prepared[saved['kind']], saved['thr']
    rows = []
    for rules in rule_sets:
        POST_RULES = set(rules)
        rows.append({'rules': '+'.join(rules) or 'жодного',
                     'val_f1': score(saved['probs']['val'], val_text, data['val']['offsets'], data['val']['locs'], thr),
                     'lab_f1': score(saved['probs']['lab'], lab_text, data['lab']['offsets'], data['lab']['locs'], thr)})
    POST_RULES = set()
    return pd.DataFrame(rows)

rule_sets = [[]] + [[r] for r in RULES] + [['newline', 'emoji', 'letters'], ['newline', 'emoji', 'strip', 'letters'], RULES]
eval_rules('word_lstm_synth', rule_sets)

In [ ]:
POST_RULES = {'newline', 'emoji', 'letters', 'upper'}
print(sorted(POST_RULES))

## Результати

In [ ]:
res = pd.DataFrame(results)
res.sort_values('val_f1', ascending=False)